In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("all_matches.csv")
df.head()

In [ ]:
pd.unique(df[['home_team','away_team']].values.ravel())

In [ ]:
from Confederations import get_confederation

all_teams = pd.unique(df[['home_team', 'away_team']].values.ravel())
unknowns = [t for t in all_teams if get_confederation(t) == "Unknown"]
print(sorted(unknowns))

In [ ]:
def get_tournament_weight(tournament, home_team, away_team):
    t = tournament.strip()
    home_conf = get_confederation(home_team)
    away_conf = get_confederation(away_team)
    conf_priority = ["UEFA", "CONMEBOL", "CONCACAF", "CAF", "AFC", "OFC", "Unknown"]
    home_rank = conf_priority.index(home_conf)
    away_rank = conf_priority.index(away_conf)
    top_conf = home_conf if home_rank <= away_rank else away_conf

    # ── Tier 4.0 ──────────────────────────────────────────
    if t == "World Cup":
        return 4.0

    # ── Tier 3.0 ──────────────────────────────────────────
    if t in ["European Championship", "Copa America", "Copa América"]:
        return 3.0

    # ── Tier 2.5 ──────────────────────────────────────────
    if t in ["European Championship qual",
             "Copa America qualifier", "Copa América qualifier",
             "African Nations Cup", "Confederations Cup", "Olympic Games","Euro Ch q & Nordic Ch","Euro Ch q & British Ch","South American Champ","Mundialito"]:
        return 2.5

    if "European Nations League" in t and "CONCACAF" not in t:
        if "A" in t: return 2.5
        elif "B" in t: return 2.0
        elif "C" in t: return 1.75
        elif "D" in t: return 1.5
        else: return 2.5

    # ── Tier 2.0 ──────────────────────────────────────────
    if t == "World Cup qualifier":
        if top_conf in ["UEFA", "CONMEBOL"]: return 2.0
        elif top_conf == "CAF":              return 1.75
        elif top_conf == "CONCACAF":         return 1.75
        elif top_conf == "AFC":              return 1.5
        else:                                return 1.25  # OFC

    if t in ["African Nations Cup qualifier",
             "World Cup q & Nordic Ch", "World Cup q & British Ch",
             "NA Champ & WC qual", "World Cup and CONCACAF Ch q",
             "World Cup and Asian Cup qual", "World Cup and African Cup qual",
             "WC and Oce Cup q", "WC q and Oce Cup"]:
        return 2.0

    # ── Tier 1.75 ─────────────────────────────────────────
    if t in ["CONCACAF Championship", "Asian Cup",
             "CONCACAF Cup", "CONCACAF Series","Intercontinental Champ"]:
        return 1.75

    if "CONCACAF Nations League" in t:
        if "A" in t: return 1.75
        elif "B" in t: return 1.5
        elif "C" in t: return 1.25
        else: return 1.75

    # ── Tier 1.5 ──────────────────────────────────────────
    if t in ["Asian Cup qualifier", "CONCACAF Ch q",
             "Gulf Cup", "Arab Cup",
             "CECAFA Cup", "COSAFA Cup"]:
        return 1.5

    if t in ["Southeast Asian Champ", "South Asian Championship",
             "West Asian Championship", "East Asian Championship",
             "Central American Cup", "CFU Championship",
             "Copa Centenario", "Caribbean Cup",
             "Caribbean Championship","CONCACAF Ch q & Car Ch","CONCACAF Ch q & Car Ch PO","North American Champ","CONCACAF Ch & Car Ch q","Balkan & C European Champ"]:
        return 1.5

    # ── Tier 1.25 ─────────────────────────────────────────
    if t in ["Oceania Nations Cup", "Oceania Nations Cup qualifier",
             "Pacific Games", "Pacific Mini Games",
             "South Pacific Games", "South Pacific Mini Games",
             "Melanesian Cup", "Polynesian Cup"]:
        return 1.25

    # ── Keyword fallbacks ─────────────────────────────────
    t_lower = t.lower()

    if "qualifier" in t_lower or "qual" in t_lower:
        return 1.5

    if any(kw in t_lower for kw in ["championship", "cup", "tournament",
                                     "games", "friendly tournament"]):
        return 1.5

    # ── Tier 1.0 — Friendlies and everything else ─────────
    return 1.0
df['weight'] = df.apply(
    lambda row: get_tournament_weight(row['tournament'], row['home_team'], row['away_team']), 
    axis=1
)

In [ ]:
df.groupby('tournament')['weight'].first().sort_values(ascending=False).tail(30)

In [ ]:
import importlib
import elo
importlib.reload(elo)
from elo import update_elo, get_elo, elo_ratings,apply_yearly_decay



In [ ]:
df_sorted = df.sort_values('date').reset_index(drop=True)
home_elos = []
away_elos = []
current_year =None

for _, row in df_sorted.iterrows():
    match_year = int(row['date'][:4])
    if current_year is None:
        current_year=match_year
    if match_year>current_year:
        apply_yearly_decay(current_year)
        current_year=match_year
    home_elos.append(get_elo(row['home_team']))
    away_elos.append(get_elo(row['away_team']))
    update_elo(
        row['home_team'], row['away_team'],
        row['home_score'], row['away_score'],
        row['weight'], row['neutral']
    )

df_sorted['home_elo'] = home_elos
df_sorted['away_elo'] = away_elos
df_sorted['elo_diff'] = df_sorted['home_elo'] - df_sorted['away_elo']

if "West Germany" in elo_ratings:
    elo_ratings["Germany"] = max(elo_ratings.get("Germany", 0), elo_ratings["West Germany"])
    del elo_ratings["West Germany"]

if "Soviet Union" in elo_ratings:
    elo_ratings["Russia"] = max(elo_ratings.get("Russia", 0), elo_ratings["Soviet Union"])
    del elo_ratings["Soviet Union"]
    
elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
print(elo_df.sort_values('elo', ascending=False).head(20))

In [ ]:
def get_recent_form(team,past_df,n=10):
    team_matches=past_df[
        (past_df['home_team']==team)|
        (past_df['away_team']==team)
        ].tail(n)
    if len(team_matches)==0:
        return 0.5
    
    results =[]
    for _, match in team_matches.iterrows():
        if match['home_team']==team:
            if match['home_score'] >match['away_score']:results.append(1.0)
            elif match['home_score'] ==match['away_score']:results.append(0.5)
            else: results.append(0.0)
        else:
            if match['home_score'] <match['away_score']:results.append(1.0)
            elif match['home_score'] ==match['away_score']:results.append(0.5)
            else: results.append(0.0)
    weights = np.exp(np.linspace(-2, 0, len(results)))
    return np.average(results, weights=weights)

In [ ]:
def get_h2h(home_team,away_team,past_df):
    h2h = past_df[
        ((past_df['home_team'] == home_team) & (past_df['away_team'] == away_team)) |
        ((past_df['home_team'] == away_team) & (past_df['away_team'] == home_team))
    ]
    if len(h2h) == 0:
        return 0.5
    wins = 0
    for _, match in h2h.iterrows():
        if match['home_team'] == home_team:
            if match['home_score'] > match['away_score']: wins += 1
        else:
            if match['away_score'] > match['home_score']:wins+=1
    return wins / len(h2h)

In [ ]:
def get_attack_rating(team,past_df,n=10):
    matches=past_df[
        (past_df['home_team'] == team) |
        (past_df['away_team'] == team)
    ].tail(n)
    if len(matches)==0:
        return 1.5
    goals = []
    for _, match in matches.iterrows():
        if match['home_team'] == team:
            goals.append(match['home_score'])
        else:
            goals.append(match['away_score'])
    weights = np.exp(np.linspace(-2, 0, len(goals)))
    return np.average(goals, weights=weights)

def get_defence_rating(team, past_df, n=10):
    matches = past_df[
        (past_df['home_team'] == team) | 
        (past_df['away_team'] == team)
    ].tail(n)
    if len(matches) == 0:
        return 1.5  # league average default
    goals_conceded = []
    for _, match in matches.iterrows():
        if match['home_team'] == team:
            goals_conceded.append(match['away_score'])
        else:
            goals_conceded.append(match['home_score'])
    weights = np.exp(np.linspace(-2, 0, len(goals_conceded)))
    return np.average(goals_conceded, weights=weights)

    

In [ ]:
df_check = pd.read_csv('matches_with_features.csv')
print(df_check.shape)
print(df_check.columns.tolist())
print(df_check.isnull().sum())
print(df_check[['attack_home', 'attack_away', 'defence_home', 'defence_away']].head(10))

In [ ]:
df = pd.read_csv('matches_with_features.csv')

df['result']=df.apply(lambda row: 2 if row['home_score']>row['away_score']
                    else(1 if row['home_score']==row['away_score']
                    else 0),axis=1)

df['home_advantage']=df['neutral'].apply(lambda x: 0 if x else 1)

df['attack_diff'] = df['attack_home'] - df['attack_away']
df['defence_diff'] = df['defence_away'] - df['defence_home']
df['form_diff'] = df['recent_form_home'] - df['recent_form_away']


features = ['elo_diff', 'form_diff', 'attack_diff', 'defence_diff',
            'h2h', 'weight', 'home_advantage']


train = df[(df['date'] >= '2000-01-01') & (df['date'] < '2018-01-01')]
test = df[df['date'] >= '2018-01-01']

X_train=train[features]
y_train=train['result']
X_test = test[features]
y_test = test['result']

print(f"Train: {len(X_train)} matches")
print(f"Test: {len(X_test)} matches")

In [ ]:
def get_team_snapshot(team, df):
    matches = df[
        (df['home_team'] == team) |
        (df['away_team'] == team)
    ].sort_values('date')

    if len(matches) == 0:
        return None

    row = matches.iloc[-1]

    if row['home_team'] == team:
        form = row['recent_form_home']
        attack = row['attack_home']
        defence = row['defence_home']
    else:
        form = row['recent_form_away']
        attack = row['attack_away']
        defence = row['defence_away']

    return {
        'elo': elo_ratings.get(team, 1500),
        'form': form,
        'attack': attack,
        'defence': defence
    }

In [ ]:
X_train = train[features]
y_train = train['result']

X_test = test[features]
y_test = test['result']

train = train.copy()

train['sample_weight'] = np.exp(
    (pd.to_datetime(train['date']).rank() / len(train)) * 2
)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Filter to neutral venue matches only for this model
neutral_train = df[(df['date'] >= '2000-01-01') & 
                   (df['date'] < '2018-01-01') & 
                   (df['neutral'] == True)]

neutral_test = df[(df['date'] >= '2018-01-01') & 
                  (df['neutral'] == True)]

features_neutral = ['elo_diff', 'form_diff', 'attack_diff', 'defence_diff', 'weight']

X_neutral_train = neutral_train[features_neutral]
y_neutral_train = neutral_train['result']
X_neutral_test = neutral_test[features_neutral]
y_neutral_test = neutral_test['result']

print(f"Neutral train: {len(X_neutral_train)}")
print(f"Neutral test: {len(X_neutral_test)}")

In [ ]:
neutral_train = neutral_train.copy()
neutral_train['sample_weight'] = np.exp(
    (pd.to_datetime(neutral_train['date']).rank() / len(neutral_train)) * 2
)



In [ ]:
# Flip every match
neutral_train_flipped = neutral_train.copy()
neutral_train_flipped['elo_diff'] = -neutral_train['elo_diff']
neutral_train_flipped['form_diff'] = -neutral_train['form_diff']
neutral_train_flipped['attack_diff'] = -neutral_train['attack_diff']
neutral_train_flipped['defence_diff'] = -neutral_train['defence_diff']
neutral_train_flipped['result'] = neutral_train['result'].map({2: 0, 0: 2, 1: 1})
neutral_train_flipped['sample_weight'] = neutral_train['sample_weight']


# Combine
neutral_train_aug = pd.concat([neutral_train, neutral_train_flipped]).reset_index(drop=True)

print(neutral_train_aug['result'].value_counts())

# Retrain
X_aug = neutral_train_aug[features_neutral]
y_aug = neutral_train_aug['result']
weights_aug = neutral_train_aug['sample_weight']

xgb_neutral = XGBClassifier(n_estimators=200, max_depth=4,
                             learning_rate=0.05, random_state=42,
                             subsample=0.8, colsample_bytree=0.8)
xgb_neutral.fit(X_aug, y_aug, sample_weight=weights_aug)

# Test
neutral_preds = xgb_neutral.predict(X_neutral_test)
print(f"Neutral accuracy: {accuracy_score(y_neutral_test, neutral_preds):.4f}")

In [ ]:
groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czechia"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

In [ ]:
# Precompute all snapshots once
wc26_teams = [team for teams in groups.values() for team in teams]
team_snapshots = {team: get_team_snapshot(team, df) for team in wc26_teams}

In [ ]:
def predict_match(team_a, team_b, weight=4.0):
    a = team_snapshots[team_a]
    b = team_snapshots[team_b]
    
    elo_diff = a['elo'] - b['elo']
    form_diff = a['form'] - b['form']
    attack_diff = a['attack'] - b['attack']
    defence_diff = b['defence'] - a['defence']
    
    fv = pd.DataFrame([[elo_diff, form_diff, attack_diff, defence_diff, weight]], 
                      columns=features_neutral)
    
    probs = xgb_neutral.predict_proba(fv)[0]
    
    return {
        f'{team_a}_win': round(float(probs[2]), 4),
        'draw': round(float(probs[1]), 4),
        f'{team_b}_win': round(float(probs[0]), 4)
    }

In [ ]:
from itertools import combinations

match_probs = {}
for team_a, team_b in combinations(wc26_teams, 2):
    match_probs[(team_a, team_b)] = predict_match(team_a, team_b)
    match_probs[(team_b, team_a)] = predict_match(team_b, team_a)

In [ ]:
def predict_match(team_a, team_b, weight=4.0):
    return match_probs[(team_a, team_b)]

In [ ]:
def simulate_match(team_a, team_b):
    probs = predict_match(team_a, team_b)

    p_a = probs[f"{team_a}_win"]
    p_draw = probs["draw"]
    p_b = probs[f"{team_b}_win"]

    total = p_a + p_draw + p_b

    p_a /= total
    p_draw /= total
    p_b /= total

    outcome = np.random.choice(
        ["A", "D", "B"],
        p=[p_a, p_draw, p_b]
    )

    return outcome, p_a, p_draw, p_b

In [ ]:
def simulate_group_match(team_a, team_b):
    outcome, _, _, _ = simulate_match(team_a, team_b)

    if outcome == "A":
        return {
            team_a: 3,
            team_b: 0
        }

    elif outcome == "B":
        return {
            team_a: 0,
            team_b: 3
        }

    else:
        return {
            team_a: 1,
            team_b: 1
        }

In [ ]:
print(simulate_match("Spain", "France"))

In [ ]:
def simulate_scoreline(team_a, team_b):
    outcome, _, _, _ = simulate_match(team_a, team_b)

    a = team_snapshots[team_a]
    b = team_snapshots[team_b]

    expected_a = max(0.3, (a['attack'] + b['defence']) / 2)
    expected_b = max(0.3, (b['attack'] + a['defence']) / 2)

    goals_a = np.random.poisson(expected_a)
    goals_b = np.random.poisson(expected_b)

    if outcome == "A":
        while goals_a <= goals_b:
            goals_a += 1

    elif outcome == "B":
        while goals_b <= goals_a:
            goals_b += 1

    else:
        goals_b = goals_a

    return goals_a, goals_b

In [ ]:
def play_group_match(team_a, team_b):
    goals_a, goals_b = simulate_scoreline(team_a, team_b)

    if goals_a > goals_b:
        pts_a, pts_b = 3, 0
    elif goals_b > goals_a:
        pts_a, pts_b = 0, 3
    else:
        pts_a, pts_b = 1, 1

    return {
        team_a: {
            "pts": pts_a,
            "gf": goals_a,
            "ga": goals_b
        },
        team_b: {
            "pts": pts_b,
            "gf": goals_b,
            "ga": goals_a
        }
    }

In [ ]:
def simulate_group(teams):

    table = {
        team: {
            "pts": 0,
            "gf": 0,
            "ga": 0
        }
        for team in teams
    }

    matches = [
        (teams[0], teams[1]),
        (teams[0], teams[2]),
        (teams[0], teams[3]),
        (teams[1], teams[2]),
        (teams[1], teams[3]),
        (teams[2], teams[3])
    ]

    for team_a, team_b in matches:

        result = play_group_match(team_a, team_b)

        for team in [team_a, team_b]:
            table[team]["pts"] += result[team]["pts"]
            table[team]["gf"] += result[team]["gf"]
            table[team]["ga"] += result[team]["ga"]

    for team in table:
        table[team]["gd"] = (
            table[team]["gf"] -
            table[team]["ga"]
        )

    standings = sorted(
        table.items(),
        key=lambda x: (
            x[1]["pts"],
            x[1]["gd"],
            x[1]["gf"]
        ),
        reverse=True
    )

    return standings

In [ ]:
def simulate_all_groups(groups):

    results = {}

    for group_name, teams in groups.items():
        results[group_name] = simulate_group(teams)

    return results

In [ ]:
group_results = simulate_all_groups(groups)

for group_name, table in group_results.items():

    print(f"\nGROUP {group_name}")

    for team, stats in table:
        print(
            f"{team:20}",
            stats["pts"],
            stats["gd"]
        )

In [ ]:
def get_qualified_teams(group_results):

    qualifiers = {}

    third_place = []

    for group_name, table in group_results.items():

        qualifiers[f"1{group_name}"] = table[0][0]
        qualifiers[f"2{group_name}"] = table[1][0]
        qualifiers[f"3{group_name}"] = table[2][0]

        third_place.append(
            (
                group_name,
                table[2][0],
                table[2][1]["pts"],
                table[2][1]["gd"],
                table[2][1]["gf"]
            )
        )

    third_place.sort(
        key=lambda x: (
            x[2],
            x[3],
            x[4]
        ),
        reverse=True
    )

    best_third = third_place[:8]

    return qualifiers, best_third

In [ ]:
R32_TEMPLATE = [
    ("2A", "2B"),
    ("1E", "3X"),

    ("1F", "2C"),
    ("1C", "2F"),

    ("1I", "3X"),
    ("2E", "2I"),

    ("1A", "3X"),
    ("1L", "3X"),

    ("1D", "3X"),
    ("1G", "3X"),

    ("2K", "2L"),
    ("1H", "2J"),

    ("1B", "3X"),
    ("1J", "2H"),

    ("1K", "3X"),
    ("2D", "2G")
]

In [ ]:
def create_round_of_32(group_results):

    qualifiers, best_third = get_qualified_teams(group_results)

    third_teams = [x[1] for x in best_third]

    # Temporary assignment of the 8 third-place slots
    third_slots = {
        "3A": third_teams[0],
        "3B": third_teams[1],
        "3C": third_teams[2],
        "3D": third_teams[3],
        "3E": third_teams[4],
        "3F": third_teams[5],
        "3G": third_teams[6],
        "3H": third_teams[7]
    }

    matches = []

    for team1_code, team2_code in R32_TEMPLATE:

        if team1_code == "3X":
            team1 = third_teams.pop(0)
        else:
            team1 = qualifiers[team1_code]

        if team2_code == "3X":
            team2 = third_teams.pop(0)
        else:
            team2 = qualifiers[team2_code]

        matches.append((team1, team2))

    return matches

In [ ]:
def simulate_knockout_match(team_a, team_b):

    probs = predict_match(team_a, team_b)

    p_a = probs[f"{team_a}_win"]
    p_draw = probs["draw"]
    p_b = probs[f"{team_b}_win"]

    total = p_a + p_draw + p_b

    p_a /= total
    p_draw /= total
    p_b /= total

    outcome = np.random.choice(
        ["A", "D", "B"],
        p=[p_a, p_draw, p_b]
    )

    if outcome == "A":
        return team_a
    elif outcome == "B":
        return team_b
    else:
        return np.random.choice([team_a, team_b])

In [ ]:
def play_knockout_round(matches):

    winners = []

    for team_a, team_b in matches:

        winner = simulate_knockout_match(team_a, team_b)

        winners.append(winner)

    return winners

In [ ]:
def simulate_tournament(groups):

    group_results = simulate_all_groups(groups)

    qualifiers, best_third = get_qualified_teams(group_results)

    # Everyone who qualified for knockouts
    r16_teams = list(qualifiers.values())

    r32_matches = create_round_of_32(group_results)

    r32_winners = play_knockout_round(r32_matches)

    qf_teams = r32_winners

    r16_matches = [
        (r32_winners[i], r32_winners[i+1])
        for i in range(0, 16, 2)
    ]

    r16_winners = play_knockout_round(r16_matches)

    sf_teams = r16_winners

    qf_matches = [
        (r16_winners[i], r16_winners[i+1])
        for i in range(0, 8, 2)
    ]

    qf_winners = play_knockout_round(qf_matches)

    final_teams = qf_winners

    sf_matches = [
        (qf_winners[0], qf_winners[1]),
        (qf_winners[2], qf_winners[3])
    ]

    sf_winners = play_knockout_round(sf_matches)

    champion = play_knockout_round(
        [(sf_winners[0], sf_winners[1])]
    )[0]

    return {
        "r16": r16_teams,
        "qf": qf_teams,
        "sf": sf_teams,
        "final": final_teams,
        "champion": champion
    }

In [ ]:
champion = simulate_tournament(groups)

print("WORLD CHAMPION:")
print(champion)

In [ ]:
from collections import Counter

r16_counter = Counter()
qf_counter = Counter()
sf_counter = Counter()
final_counter = Counter()
champion_counter = Counter()


In [ ]:
N = 5000

for _ in range(N):

    result = simulate_tournament(groups)

    for team in result["r16"]:
        r16_counter[team] += 1

    for team in result["qf"]:
        qf_counter[team] += 1

    for team in result["sf"]:
        sf_counter[team] += 1

    for team in result["final"]:
        final_counter[team] += 1

    champion_counter[result["champion"]] += 1

In [ ]:
results = []

all_teams = sorted(r16_counter.keys())

for team in all_teams:

    results.append({
        "Team": team,
        "R16": round(100 * r16_counter[team] / N, 2),
        "QF": round(100 * qf_counter[team] / N, 2),
        "SF": round(100 * sf_counter[team] / N, 2),
        "Final": round(100 * final_counter[team] / N, 2),
        "Champion": round(100 * champion_counter[team] / N, 2)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Champion",
    ascending=False
)

results_df.head(20)